# HW4 C: Text-to-speech with a speech-token language model (40 automatic + 3 report points)
Implement the original 12-layer causal Transformer that maps text tokens to speech tokens.
The frozen CosyVoice2 flow model and vocoder convert those tokens to audio.
This is the homework model inspired by CosyVoice2, not a reproduction of its pretrained
Qwen language model. Train on the full original pre-tokenized LibriTTS training cache.

**Grading:** dataset (2), collate (2), teacher-forcing sequence (4), forward loss and
causality (3), generation (2), training (2), validation (1), and synthesized-speech
WER (24). The 16 implementation points do not depend on model convergence.
Performance points reward improvement over the supplied starter:

| Synthesized-speech WER | Performance |
|---|---:|
| Below 15% | 24 |
| 15% to below 17% | 20 |
| 17% to below 19% | 16 |
| 19% to below 21% | 12 |
| 21% to below 22.5% | 8 |
| 22.5% to below 23.5% | 4 |
| 23.5% or higher | 0 |

The original three-epoch reference reached 14.56%; the two-epoch starter reached
23.55% on the same 200 texts and voice prompt. The zero-credit band includes the
rounded 24% starting baseline. Missing predictions count as deletions.
Answer report question C1 in the shared PDF.

**Written answers:** submit one searchable, text-selectable `report.pdf` using the
five section IDs **A1, A2, A3, B1, C1** in `report_template.txt`. Put conceptual
answers and requested figures in that PDF; notebook markdown answers are not
manually graded. Aim for 3–5 sentences per section, plus the requested figures or
token sequence. There is no minimum page count. No extra experiments are required
for B1/C1. Mark unfinished work `Not completed` and explain what is available.
The shared report is worth 15 points (A: 9, B: 3, C: 3); the autograder is worth 85.


## Setup
Open this notebook on Colab and select a GPU. Setup automatically downloads the
small support bundle from the public course Hugging Face repository; no account,
token, or TA Drive access is needed. The next cell downloads only this part's data.
Downloads use a fixed release and verify checksums. Setup installs Python
dependencies while preserving Colab's matched PyTorch/CUDA installation.
Run setup once in a fresh runtime. If pip asks for a restart, restart and rerun setup.

Implement each TODO before running the cells that use it. For a quick check of
**your implementation**, set `SMOKE_TEST = True`. Smoke results are not completed experiments.
Definitions used by the autograder must be top-level functions/classes with the
specified names and signatures. The grader provides the standard imports shown
in setup; put any additional imports inside your functions. Keep helper constants
inside functions or as class attributes: top-level assignments are not executed.
A syntax error skips that cell; other independently implemented questions still count.

### What you must complete

| Stage | Your implementation | Provided scaffolding / completion check |
|---|---|---|
| Data | `CosyVoiceDataset.__getitem__`, `cosyvoice_collate_fn` | Cache loading and loaders; padded token tensors and matching valid lengths |
| Model | `prepare_sequence`, `_logits`, `forward`, `generate` | Exact constructor and starter loader; shifted targets, causal logits, finite scalar loss, speech-token output |
| Training | `train_epoch`, `validate` | Optimizer defaults, scheduler, epoch loop, best/latest checkpoints and resume |
| Evaluation | Run the supplied decoding, voice-prompt, and evaluation cells | Audio synthesis, Whisper transcription, resumable prediction export |

Keep the provided original **768-width / 12-layer / 12-head** constructor unchanged
so the starter loads exactly. In smoke mode, check a few batches without the
starter: loss/accuracy must be finite, a training update must change parameters,
and validation must leave them unchanged. Check the worked sequence example below
before training. Then set `SMOKE_TEST=False`, load the course two-epoch starter,
and run the supplied default of one additional student epoch (or save your
completed work if compute ends sooner). Optimizer tuning is optional, using
validation results; implementing the per-batch training loop is required.

After training, load the best validation checkpoint, verify ground-truth token
decoding, supply your 3–10 second voice prompt with its exact transcript, and run
all **200** evaluation texts. The evaluator writes `RESULTS/tts_submission.json`.
Its transcriptions must come from Whisper applied to your synthesized audio;
do not replace them with target text or manually correct recognition errors.
Audio and checkpoints remain on your Drive/local runtime. Only predictions,
your notebook, and the shared report go into the ZIP. Answer C1 in the PDF.


In [ ]:
REQUIREMENTS = 'requirements-tts.txt'
# Colab: select Runtime > Change runtime type > GPU before training.
import sys, subprocess
from pathlib import Path

# Course release settings are supplied by staff; no student login is needed.
COURSE_REPO_ID = 'Louis0324/eecs183-hw4'
COURSE_REVISION = 'hw4-2026-v1'
import os
os.environ.setdefault("HW4_HF_REPO", COURSE_REPO_ID)
os.environ.setdefault("HW4_HF_REVISION", COURSE_REVISION)

def ensure_hw4_support(repo_id=None, revision='hw4-2026-v1', packager_only=False):
    """Fetch the public support bundle, or use a local/uploaded offline copy."""
    import hashlib
    from io import BytesIO
    import json
    from pathlib import Path
    import re
    from urllib.parse import quote
    from urllib.request import urlopen
    import zipfile

    required = 'make_submission.py' if packager_only else 'course_download.py'
    if Path(required).is_file():
        return
    local_bundle = Path('hw4-student.zip')
    if local_bundle.is_file():
        content = local_bundle.read_bytes()
    elif repo_id:
        if not re.fullmatch(r'[\w.-]+/[\w.-]+', repo_id):
            raise ValueError('Expected a Hugging Face repository ID like COURSE_ORG/eecs183-hw4.')
        base = f'https://huggingface.co/datasets/{repo_id}/resolve/{quote(revision, safe="")}/'
        print('Downloading the small HW4 support bundle from Hugging Face; no login is required.')
        try:
            with urlopen(base + 'release_manifest.json', timeout=30) as response:
                manifest = json.loads(response.read(1024 * 1024))
            spec = manifest['files']['hw4-student.zip']
            if not 0 < spec['bytes'] < 30 * 1024 * 1024:
                raise ValueError('Invalid support bundle size in the release manifest.')
            with urlopen(base + 'hw4-student.zip', timeout=30) as response:
                content = response.read(spec['bytes'] + 1)
            if len(content) != spec['bytes'] or hashlib.sha256(content).hexdigest() != spec['sha256']:
                raise ValueError('Support bundle checksum mismatch; rerun to download a fresh copy.')
        except Exception as error:
            raise RuntimeError('Could not download the HW4 support release. Rerun this cell; '
                               'if needed, download hw4-student.zip from the course Hugging Face repo '
                               'and upload it through the Files sidebar before retrying.') from error
    else:
        # Allows local staff testing and offline/manual use before publication.
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError('Put hw4-student.zip in this folder or configure the course Hugging Face repository.') from error
        print('Upload hw4-student.zip to load the supplied helpers.')
        uploaded = files.upload()
        archives = [value for name, value in uploaded.items() if name.lower().endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('Upload exactly one hw4-student.zip support bundle, then rerun.')
        content = archives[0]

    with zipfile.ZipFile(BytesIO(content)) as archive:
        if archive.testzip() is not None:
            raise ValueError('Corrupt support ZIP; download a fresh copy.')
        for item in archive.infolist():
            path = Path(item.filename)
            if (path.is_absolute() or '..' in path.parts or
                    ((item.external_attr >> 16) & 0o170000) == 0o120000 or
                    not path.resolve().is_relative_to(Path.cwd().resolve())):
                raise ValueError('Unsafe support archive path.')
            # Browser/local notebook edits must never be replaced by starter copies.
            if item.is_dir() or path.suffix == '.ipynb' or (packager_only and item.filename != required):
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(item))
    if not Path(required).is_file():
        raise ValueError(f'This ZIP is missing {required}; use the current hw4-student.zip support bundle.')

ensure_hw4_support(COURSE_REPO_ID, COURSE_REVISION)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', REQUIREMENTS])
import torch
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SMOKE_TEST = False  # True: only a few steps, for debugging your implementation.
DATA_ROOT = Path(os.environ.get('HW4_DATA_ROOT', 'data'))
RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
# Checkpoints/results disappear when Colab deletes a runtime. Copy them to Drive
# or download them before disconnecting. Submission ZIPs contain no checkpoints.

In [ ]:
# Download only Part C data; verified cached copies are reused.
from course_download import ensure_course_data
DATA_ARCHIVE = None  # Optional offline override: Path("hw4-data-c.zip")
ensure_course_data('c', data_root=DATA_ROOT, archive=DATA_ARCHIVE)


In [ ]:
import json, math, copy
import numpy as np
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence, unpad_sequence
from sklearn.metrics import accuracy_score, r2_score
import matplotlib.pyplot as plt
from hw4_runtime import load_phoneme_data, load_mfcc_data, load_audio
np.random.seed(183); torch.manual_seed(183)
device = DEVICE

from IPython.display import Audio, display
class Config:
    IGNORE_ID = -1
    SOS_EOS_ID = 0  # Special-embedding index; NOT speech EOS.
    TASK_ID = 1    # Special-embedding index; NOT a speech-token offset.
    DATA_CACHE_DIR = str(DATA_ROOT / 'tts')
    PRETRAINED_DIR = os.environ.get('HW4_PRETRAINED_DIR', 'pretrained_models')
    RESULTS_DIR = str(RESULTS)

## 1. Load training data and the text tokenizer
Training uses cached speech tokens, so the speech tokenizer, flow model and vocoder
are loaded only for audio inference. This avoids keeping unnecessary models on the GPU.
The text vocabulary includes added tokens: use the tokenizer's full vocabulary size.

In [ ]:
from hw4_util import download_pretrained_models, load_text_tokenizer
model_dir = download_pretrained_models(Config.PRETRAINED_DIR, tokenizer_only=True)
text_tokenizer = load_text_tokenizer(model_dir)
train_cache = torch.load(DATA_ROOT / 'tts/libritts_train.pt', map_location='cpu', weights_only=True)
val_cache = torch.load(DATA_ROOT / 'tts/libritts_dev.pt', map_location='cpu', weights_only=True)
train_data, val_data = train_cache['samples'], val_cache['samples']
speaker_to_idx = train_cache['metadata']['speaker_to_idx']
speech_vocab_size = train_cache['metadata']['speech_vocab_size']
print('Train:', len(train_data), 'Validation:', len(val_data), 'Speech vocabulary:', speech_vocab_size)

## 2. Dataset and batching
Samples contain `utt_id`, `text`, `speaker_id`, and `speech_tokens`.
Return long tensors for token IDs. Return `None` for empty/invalid samples or
examples exceeding either length limit. Do not independently truncate the text
and speech: that can make them describe different content. Unseen validation
speakers use `speaker_idx=-1`; speaker indices are metadata in this model.
The collator filters `None`, sorts by decreasing speech length, pads with zero,
and returns the fields listed in its docstring. An all-invalid batch returns `None`.
The default limits (512 text tokens, 1,200 speech tokens) accommodate the full
original cache. Use the full supplied training set for the experiment; if your
compute budget ends early, save the checkpoint and report the training completed.
The small subset used when `SMOKE_TEST=True` is only for checking your implementation.


In [ ]:
class CosyVoiceDataset(Dataset):

    def __init__(self, samples_list, text_tokenizer, speaker_to_idx_dict, max_text_len=512, max_speech_len=1200):
        self.samples = samples_list
        self.text_tokenizer = text_tokenizer
        self.speaker_to_idx = speaker_to_idx_dict
        self.max_text_len = max_text_len
        self.max_speech_len = max_speech_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        """Return dict: utt, text, text_token (long [T]), speech_token (long [S]), speaker_idx, speaker_id. Follow the validation and maximum-length rules above. Preserve the audio/text pairing."""
        raise NotImplementedError('TODO: __getitem__')

In [ ]:
def cosyvoice_collate_fn(batch):
    """Return None for an all-invalid batch; otherwise sort by speech length descending and return utts, text, text_tokens, text_lengths, speech_tokens, speech_lengths, speaker_indices. Pad token tensors with 0; return lengths and indices as int64. Reorder every field consistently."""
    raise NotImplementedError('TODO: cosyvoice_collate_fn')

In [ ]:
train_dataset = CosyVoiceDataset(train_data[:16] if SMOKE_TEST else train_data, text_tokenizer, speaker_to_idx)
val_dataset = CosyVoiceDataset(val_data[:8] if SMOKE_TEST else val_data, text_tokenizer, speaker_to_idx)
train_loader = DataLoader(train_dataset, batch_size=2 if SMOKE_TEST else 4, shuffle=True, num_workers=0, collate_fn=cosyvoice_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=2 if SMOKE_TEST else 4, shuffle=False, num_workers=0, collate_fn=cosyvoice_collate_fn)
first_batch = next(x for x in train_loader if x is not None)
print({k: tuple(v.shape) for k, v in first_batch.items() if torch.is_tensor(v)})

## 3. Causal speech-token LM
Use separate embeddings for text tokens, speech tokens, and the two special tokens.
Speech IDs are `0..speech_vocab_size-1`; the **speech EOS** is `speech_vocab_size`.
The special SOS embedding has index 0 in a different embedding table.

For text `[t0,t1]` and speech `[s0,s1,s2]`:

- Input embeddings: `[SOS,t0,t1,TASK,s0,s1,s2]`
- Targets: `[-1,-1,-1,s0,s1,s2,EOS]`

The TASK position predicts the first speech token. The final speech input predicts
EOS. Ignore text-prefix and padded targets with `IGNORE_ID=-1`. Attention uses a
causal mask and a padding mask; `True` means blocked/padded. Return loss and accuracy
over non-ignored targets only. Provide `_logits(inputs, padding=None)` so causality
and generation can be tested independently. It must add positions, apply causal
attention, and return raw `[B,T,speech_vocab_size+1]` logits.

Use the original architecture: **d_model=768, 12 heads, 12 layers**, with learned
positions and max_seq_len=2048. `SMOKE_TEST` reduces the number of examples and
updates; it keeps this architecture. The constructor must still honor dimension
arguments so the autograder can run small isolated unit checks on CPU.
**Provided constructor:** keep `__init__` and its module names unchanged. It fixes
the checkpoint format: feed-forward width `4*d_model`, ReLU, dropout 0, post-norm,
and separate learned embeddings. You implement the methods below it. The course
starter loader checks the architecture, tokenizer IDs, weight shapes, and file
checksum before loading. Successful loading does not validate your forward or
generation logic; run the implementation checks too.


In [ ]:
class TextToSpeechLM(nn.Module):

    def __init__(self, text_vocab_size, speech_vocab_size, d_model=768, n_heads=12, n_layers=12, max_seq_len=2048):
        super().__init__()
        self.model_config = dict(d_model=d_model, n_heads=n_heads, n_layers=n_layers, max_seq_len=max_seq_len)
        self.d_model = d_model
        self.speech_vocab_size = speech_vocab_size
        self.text_embedding = nn.Embedding(text_vocab_size, d_model)
        self.speech_embedding = nn.Embedding(speech_vocab_size + 1, d_model)
        self.special_embedding = nn.Embedding(2, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4, dropout=0.0, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.output_projection = nn.Linear(d_model, speech_vocab_size + 1)

    def prepare_sequence(self, text_tokens, text_lengths, speech_tokens=None, speech_lengths=None):
        """Inputs: padded text/speech IDs and valid lengths. Return input embeddings [B,L,D], int64 targets [B,L], bool padding mask [B,L]. Follow the exact shift described above. If speech is None, return the text prefix and all ignored targets. Preserve embedding gradients."""
        raise NotImplementedError('TODO: prepare_sequence')

    def _logits(self, inputs, padding=None):
        """Add learned position embeddings. Apply a boolean upper-triangular causal mask and optional boolean padding mask to the Transformer. Return raw logits [B,L,V+1]. Reject sequences longer than max_seq_len."""
        raise NotImplementedError('TODO: _logits')

    def forward(self, text_tokens, text_lengths, speech_tokens, speech_lengths):
        """Call prepare_sequence and _logits; compute scalar cross entropy with ignore_index=-1 and scalar accuracy on non-ignored targets. Return (loss, accuracy)."""
        raise NotImplementedError('TODO: forward')

    @torch.no_grad()
    def generate(self, text_tokens, max_length=500, temperature=1.0, top_k=50, prompt_speech_token=None, prompt_text_tokens=None, min_length=None):
        """Batch size 1. Set eval mode and disable gradients. Prefix: SOS, prompt text (if any), target text, TASK, prompt speech (if any). Repeatedly call _logits, sample using temperature>0 and top-k (clamp k to vocabulary size; 0 disables top-k). Block EOS before min_length; stop at EOS or max_length new tokens. Return long [1,N] containing ONLY new speech tokens, excluding prompt and EOS. EOS ID is speech_vocab_size."""
        raise NotImplementedError('TODO: generate')


In [ ]:
# Keep the original architecture in smoke mode; only reduce the data/steps.
MODEL_KWARGS = dict(d_model=768, n_heads=12, n_layers=12, max_seq_len=2048)
model = TextToSpeechLM(text_tokenizer.vocab_size, speech_vocab_size, **MODEL_KWARGS).to(device)
print('Trainable parameters:', sum(p.numel() for p in model.parameters()))
# Use the course starter for normal training; smoke mode needs no checkpoint download.
# For your own interrupted training, set RESUME=True and keep the same RESULTS directory.
RESUME = False
USE_STARTER = not SMOKE_TEST
STARTER_CHECKPOINT = os.environ.get('HW4_TTS_STARTER')  # Optional local file override.
starter_metadata = None
if USE_STARTER and not RESUME:
    from tts_checkpoint import load_tts_starter
    from course_download import download_course_file
    STARTER_CHECKPOINT = download_course_file("hw4-tts-starter.pt", local_path=STARTER_CHECKPOINT)
    starter_metadata = load_tts_starter(model, text_tokenizer, STARTER_CHECKPOINT)
    print('Loaded course starter; completed reference epochs:',
          starter_metadata['source_training']['completed_epochs'])


## 4. Training and validation
Implement the two functions below. Skip `None` batches. Training must set train
mode, clear gradients, backpropagate, clip gradients, step the optimizer, and step
the scheduler once per valid batch. Validation must set eval mode, disable gradients,
and leave parameters unchanged. Return mean loss and mean accuracy over valid batches;
return `(0.0, 0.0)` if no valid batches were present.
The model architecture stays fixed. Reduce batch size if memory is limited.
Resume only checkpoints with the supplied architecture and the same training
settings. Use a fresh results directory when starting a different experiment.

The model-loading cell downloads `hw4-tts-starter.pt` automatically from the
public course Hugging Face repository (about 407 MiB) and checks its checksum.
An optional `STARTER_CHECKPOINT` path can reuse an existing local copy.
It contains the original model after two full reference epochs. Storage is FP16
to reduce download size; loading copies weights into the normal FP32 model.
Create a fresh optimizer for this warm start. The default is **one additional
student epoch**, and you can stop earlier if your compute budget ends. We retain
the entire original dataset. `USE_STARTER=False` keeps training from scratch
available; `SMOKE_TEST=True` skips the starter download. For your own saved
training state use `RESUME=True`, which restores your optimizer and scheduler
instead of reloading the course starter. `num_epochs` counts student epochs only.

The staff implementation measured 23.55% WER from the two-epoch starter, compared
with 14.56% after the third reference epoch. These used the same fixed voice prompt
and 200 texts. They show room for improvement, but your prompt, implementation,
batch size, and fresh optimizer can produce different results.


**Provided training scaffolding:** the optimizer, learning-rate scheduler, outer
loop over epochs, checkpoint saving/resumption, and validation-based checkpoint
selection are supplied. The defaults are AdamW with learning rate `2e-4`, weight
decay `0.01`, and 10% warmup followed by cosine decay. Use these as a starting
point; you may tune them based on validation results. Changing optimizer settings
does not affect loading the course starter weights. When resuming your own
training state, keep the original training settings and model constructor.

**Your implementation:** write `train_epoch` and `validate` below. The per-batch
training loop, device transfers, gradient clearing, backpropagation, gradient
clipping, optimizer/scheduler steps, and metric aggregation are your work.
Validation must disable gradients and leave model parameters unchanged.


In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, device):
    """Implement one epoch with the exact behavior above. Move text_tokens, text_lengths, speech_tokens, speech_lengths to device. scheduler may be None. Return (mean_loss, mean_accuracy)."""
    raise NotImplementedError('TODO: train_epoch')

In [ ]:
def validate(model, dataloader, device):
    """Evaluate valid batches with model.eval() and torch.no_grad(). No updates or gradients. Return (mean_loss, mean_accuracy), or (0.0,0.0) for no valid batches."""
    raise NotImplementedError('TODO: validate')

In [ ]:
from hw4_util import get_warmup_cosine_scheduler, save_checkpoint
if USE_STARTER and not RESUME and starter_metadata is None:
    raise RuntimeError('Load the course starter successfully before training, or set USE_STARTER=False.')
num_epochs = 1 if (SMOKE_TEST or USE_STARTER) else 3
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
total_steps = num_epochs * len(train_loader)
scheduler = get_warmup_cosine_scheduler(optimizer, max(1, total_steps // 10), total_steps)
best_val_loss, start_epoch = float('inf'), 0
if RESUME:
    if not (RESULTS / 'tts_latest.pt').exists():
        raise FileNotFoundError('No tts_latest.pt in RESULTS; restore your saved checkpoint or set RESUME=False.')
    state = torch.load(RESULTS / 'tts_latest.pt', map_location='cpu', weights_only=True)
    if state.get('model_config') != MODEL_KWARGS:
        raise ValueError('Resume checkpoint architecture does not match the original TTS model.')
    model.load_state_dict(state['model_state_dict'], strict=True)
    starter_metadata = state.get('starter_metadata')
    optimizer.load_state_dict(state['optimizer_state_dict'])
    if state.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(state['scheduler_state_dict'])
    start_epoch = state['epoch'] + 1
    if (RESULTS / 'tts_best.pt').exists():
        best_val_loss = torch.load(RESULTS / 'tts_best.pt', map_location='cpu', weights_only=True)['val_loss']
for epoch in range(start_epoch, num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc = validate(model, val_loader, device)
    print(f'Epoch {epoch+1}: train={train_loss:.4f}, dev={val_loss:.4f}, dev accuracy={val_acc:.2%}')
    save_checkpoint(model, optimizer, scheduler, epoch, train_loss, val_loss, RESULTS / 'tts_latest.pt', model_config=MODEL_KWARGS, starter_metadata=starter_metadata)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, None, None, epoch, train_loss, val_loss, RESULTS / 'tts_best.pt', model_config=MODEL_KWARGS, starter_metadata=starter_metadata)


## 5. Verify audio decoding and synthesize
First decode ground-truth tokens. This separates codec/setup errors from LM errors.
The pretrained components require additional model downloads; no training occurs here.
The helpers raise errors on missing dependencies or incompatible checkpoints.
After saving checkpoints, the code frees training optimizer states before loading
the audio models. Rerun the training setup cell to create an optimizer if you
return to training later.


In [ ]:
# Checkpoints are saved. Release training state before loading the audio models.
import gc
model.zero_grad(set_to_none=True)
globals().pop('scheduler', None)
globals().pop('optimizer', None)
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()

from hw4_util import load_speech_tokenizer, load_flow_model, load_vocoder, generate_audio_from_tokens, synthesize
model_dir = download_pretrained_models(Config.PRETRAINED_DIR)
speech_tokenizer = load_speech_tokenizer(model_dir, device)
flow_model = load_flow_model(model_dir, device)
vocoder = load_vocoder(model_dir, device)
sample = next(x for x in (val_dataset[i] for i in range(len(val_dataset))) if x is not None)
audio, mel = generate_audio_from_tokens(sample['speech_token'][:50] if SMOKE_TEST else sample['speech_token'], flow_model, vocoder, device)
display(Audio(audio.cpu().numpy(), rate=24000))
state = torch.load(RESULTS / 'tts_best.pt', map_location='cpu', weights_only=True)
model.load_state_dict(state['model_state_dict']); model.eval()
audio, sr = synthesize('This is a short speech synthesis test.', model, text_tokenizer, speech_tokenizer, flow_model, vocoder, device,
                       output_path=RESULTS / 'tts_example.wav', max_length=20 if SMOKE_TEST else 500)
display(Audio(audio.numpy(), rate=sr))

## 6. Voice prompting and final evaluation
Record 3–10 seconds of your own speech and supply its exact transcript. The upload
helper supports WAV/FLAC and resamples to mono 16 kHz. A local file can instead be
passed as `audio_path`. The model conditions on prompt text and speech tokens; the
flow model also uses the prompt's speaker embedding and mel features.

Evaluate the 200 provided target texts, transcribe synthesized audio with Whisper,
and save predictions using the original numeric text IDs. This ordering needs no
student-ID shuffle. ID shuffling does not establish authorship. WER measures
intelligibility rather than speaker similarity; C1 asks about this distinction.
The evaluator saves each result and resumes incomplete work. It also keeps audio
locally for listening; audio files are not included in the submission ZIP.
Matching model weights resume existing predictions. Set `RESTART_PREDICTIONS=True`
if you changed decoding/generation code or settings and want to reevaluate from
the beginning; set it back to `False` to resume an interrupted evaluation.


In [ ]:
from hw4_util import record_or_load_voice, run_voice_cloning_evaluation
prompt_audio, prompt_text, recording_path = record_or_load_voice(
    prompt_text='The quick brown fox jumps over the lazy dog.', results_dir=RESULTS)
STUDENT_ID = 'YOUR_STUDENT_ID'  # Replace this; no ID-based shuffling is performed.
RESTART_PREDICTIONS = False  # True after changing generation code/settings.
submission_path = run_voice_cloning_evaluation(
    STUDENT_ID, model, text_tokenizer, speech_tokenizer, flow_model, vocoder,
    prompt_audio, prompt_text, results_dir=RESULTS, pretrained_dir=Config.PRETRAINED_DIR,
    device=device, model_dir=model_dir, max_examples=2 if SMOKE_TEST else None, restart=RESTART_PREDICTIONS)
print('Saved', submission_path)

## Report C1: Teacher forcing, masking, and WER (3 manual points)
Put this answer in the **C1** section of `report.pdf`. Use about 3–5 sentences plus
the two token sequences. This question needs no additional training or synthesis.

1. For text `[t0]` and speech `[s0, s1]`, write the input-embedding sequence and
   target sequence, including SOS, TASK, EOS, and ignored targets. Explain which
   position predicts the first speech token. (1 point)
2. Explain what information would leak without the causal attention mask, and why
   padded positions must also be excluded. (1 point)
3. Explain why low synthesized-speech WER does not establish that the generated
   voice sounds like the prompt speaker. (1 point)


## Prepare your submission
**Default: submit one combined ZIP containing one shared `report.pdf`.**
The PDF covers **A1, A2, A3, B1, C1** (and BONUS if attempted). Use
`report_template.txt`, keep all five headings, and write `Not completed` only for
answers you did not finish. You may answer B1/C1 without training those models.
Export one PDF with selectable text from Google Docs, Word, or LaTeX. If you wrote
separate reports, combine them into this document before uploading; the helper
**does not merge PDFs**. Notebook PDFs are not required.

**Typical A → B → C workflow**

1. **After A:** run its final cell. Download `submission.zip` and rename it
   `hw4-a.zip` on your computer. At the report prompt, upload your current combined
   PDF or cancel if it is not ready.
2. **After B:** leave `include_previous=True` (B's default). At the **earlier ZIP**
   prompt, upload `hw4-a.zip`. At the separate **report** prompt, upload an updated
   combined PDF, or cancel to keep the report already in the ZIP. The cell adds
   your current B notebook and available B predictions. Save the download as
   `hw4-ab.zip`.
3. **After C:** leave `include_previous=True` (C's default). Upload `hw4-ab.zip`
   at the earlier-ZIP prompt, then your final combined `report.pdf` at the report
   prompt. The cell adds C and downloads the final `submission.zip`. Upload
   **that one ZIP** to the HW4 Programming Assignment on Gradescope.

**Stopping early or working in another order**

| Work you want graded | What to do |
|---|---|
| A + B only | Finish step 2 with your final report, then submit `hw4-ab.zip`. You do **not** need to open or run C. |
| A only, or part of A | Submit A's ZIP with the report answers you completed. |
| Only B or C | Cancel the earlier-ZIP prompt; the cell still packages the current notebook and available results. |
| Separate A and B ZIPs, finishing C | Select both A and B ZIPs together at C's earlier-ZIP prompt, then upload the shared report at the next prompt. |
| Another order, or updating earlier work | Set `include_previous=True` in whichever notebook you package last and import your latest combined ZIP. Current notebook edits and available local results replace their earlier copies. |

Use the **latest** ZIP for each part; do not select both an old and a new version
of the same part. Keep `include_report=True`: cancelling its upload preserves an
existing report; setting it to `False` deliberately leaves the report out.
Missing code, predictions, or a PDF do not block grading of other completed work.
Missing C does not reduce A/B credit. Save each ZIP before disconnecting Colab.

**Before uploading:** read the printed notebook summary, file list, and warnings.
Check that all notebooks you want graded and their available prediction files
are included. Open `report.pdf` inside the ZIP to confirm it is your latest report.
Warnings for intentionally unfinished questions are expected; fix warnings about
work you intended to include. Every Gradescope resubmission must contain **all**
work you want graded; earlier attempts are not combined with the newest one.

**Packaging after a runtime reset:** you can run just the final cell; it downloads
the small support bundle from Hugging Face only if its helper is missing. No GPU, dataset download,
dependency installation, or training is needed for packaging. Set
`include_previous=True` and import your saved submission ZIP to recover earlier
results. If newer results exist only on Drive, mount it and restore your `RESULTS`
path first; the helper cannot recover files from a deleted runtime.

C performance uses `results/tts_submission.json` (200 test IDs for a complete run).

The ZIP contains notebooks, available small predictions/plots, and the shared PDF.
It excludes checkpoints, pretrained weights, datasets, and audio. Keep those
separately for resuming training; do not upload them or the support bundle to
Gradescope. Locally, save your edited notebooks, `results/`, and `report.pdf`
together, then run `python make_submission.py` in that folder.


In [ ]:
include_report = True  # Upload one combined PDF, or cancel to retain an earlier one.
include_previous = True  # Merge earlier submission ZIPs; cancel if none.

# Packaging needs no training dependencies, GPU, or data download.
# Course release settings are supplied by staff; no student login is needed.
COURSE_REPO_ID = 'Louis0324/eecs183-hw4'
COURSE_REVISION = 'hw4-2026-v1'
import os
os.environ.setdefault("HW4_HF_REPO", COURSE_REPO_ID)
os.environ.setdefault("HW4_HF_REVISION", COURSE_REVISION)

def ensure_hw4_support(repo_id=None, revision='hw4-2026-v1', packager_only=False):
    """Fetch the public support bundle, or use a local/uploaded offline copy."""
    import hashlib
    from io import BytesIO
    import json
    from pathlib import Path
    import re
    from urllib.parse import quote
    from urllib.request import urlopen
    import zipfile

    required = 'make_submission.py' if packager_only else 'course_download.py'
    if Path(required).is_file():
        return
    local_bundle = Path('hw4-student.zip')
    if local_bundle.is_file():
        content = local_bundle.read_bytes()
    elif repo_id:
        if not re.fullmatch(r'[\w.-]+/[\w.-]+', repo_id):
            raise ValueError('Expected a Hugging Face repository ID like COURSE_ORG/eecs183-hw4.')
        base = f'https://huggingface.co/datasets/{repo_id}/resolve/{quote(revision, safe="")}/'
        print('Downloading the small HW4 support bundle from Hugging Face; no login is required.')
        try:
            with urlopen(base + 'release_manifest.json', timeout=30) as response:
                manifest = json.loads(response.read(1024 * 1024))
            spec = manifest['files']['hw4-student.zip']
            if not 0 < spec['bytes'] < 30 * 1024 * 1024:
                raise ValueError('Invalid support bundle size in the release manifest.')
            with urlopen(base + 'hw4-student.zip', timeout=30) as response:
                content = response.read(spec['bytes'] + 1)
            if len(content) != spec['bytes'] or hashlib.sha256(content).hexdigest() != spec['sha256']:
                raise ValueError('Support bundle checksum mismatch; rerun to download a fresh copy.')
        except Exception as error:
            raise RuntimeError('Could not download the HW4 support release. Rerun this cell; '
                               'if needed, download hw4-student.zip from the course Hugging Face repo '
                               'and upload it through the Files sidebar before retrying.') from error
    else:
        # Allows local staff testing and offline/manual use before publication.
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError('Put hw4-student.zip in this folder or configure the course Hugging Face repository.') from error
        print('Upload hw4-student.zip to load the supplied helpers.')
        uploaded = files.upload()
        archives = [value for name, value in uploaded.items() if name.lower().endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('Upload exactly one hw4-student.zip support bundle, then rerun.')
        content = archives[0]

    with zipfile.ZipFile(BytesIO(content)) as archive:
        if archive.testzip() is not None:
            raise ValueError('Corrupt support ZIP; download a fresh copy.')
        for item in archive.infolist():
            path = Path(item.filename)
            if (path.is_absolute() or '..' in path.parts or
                    ((item.external_attr >> 16) & 0o170000) == 0o120000 or
                    not path.resolve().is_relative_to(Path.cwd().resolve())):
                raise ValueError('Unsafe support archive path.')
            # Browser/local notebook edits must never be replaced by starter copies.
            if item.is_dir() or path.suffix == '.ipynb' or (packager_only and item.filename != required):
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(item))
    if not Path(required).is_file():
        raise ValueError(f'This ZIP is missing {required}; use the current hw4-student.zip support bundle.')

ensure_hw4_support(COURSE_REPO_ID, COURSE_REVISION, packager_only=True)
try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None
from make_submission import build_submission, from_colab
if colab_files is not None:
    from_colab('c', include_report=include_report, include_previous=include_previous,
               results_dir=globals().get("RESULTS", "results"))
else:
    build_submission(report="report.pdf" if include_report else None)
